# 00 — Data Exploration

Exploratory analysis of the synthetic customer-support dataset.

**Goals**
- Understand distributions of all variables
- Check treatment/control balance before any matching
- Identify confounding patterns

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.generate import generate_synthetic_conversations
from src.analysis.balance import balance_table

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load data

In [ ]:
# Generate fresh or load from CSV if it already exists
import os
CSV = '../data/synthetic_conversations.csv'
if os.path.exists(CSV):
    df = pd.read_csv(CSV, parse_dates=['created_at'])
else:
    df = generate_synthetic_conversations(n=2000, seed=42)

print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T.round(2)

## 2. Treatment assignment

In [ ]:
print('Overall AI-assisted rate:', df['ai_assisted'].mean().round(3))
print()
print('By period:')
print(df.groupby('period')['ai_assisted'].mean().rename({0: 'pre', 1: 'post'}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

df['ai_assisted'].value_counts().plot.bar(ax=axes[0], color=['#4C72B0', '#DD8452'])
axes[0].set_title('Treatment counts')
axes[0].set_xticklabels(['Human only', 'AI assisted'], rotation=0)

df.groupby(['period', 'ai_assisted']).size().unstack().plot.bar(
    ax=axes[1], color=['#4C72B0', '#DD8452']
)
axes[1].set_title('Treatment by period')
axes[1].set_xticklabels(['Pre', 'Post'], rotation=0)
axes[1].legend(['Human only', 'AI assisted'])

plt.tight_layout()
plt.show()

## 3. Outcome distributions

In [ ]:
outcomes = ['resolution_time', 'satisfaction_score', 'escalated']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, outcomes):
    for treat, grp in df.groupby('ai_assisted'):
        label = 'AI assisted' if treat else 'Human only'
        grp[col].hist(ax=ax, alpha=0.5, bins=30, label=label, density=True)
    ax.set_title(col)
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Unadjusted mean outcomes by treatment
df.groupby('ai_assisted')[outcomes].mean().rename(index={0: 'Human only', 1: 'AI assisted'}).round(3)

## 4. Covariate distributions

In [ ]:
covariates = ['issue_severity', 'customer_tenure', 'time_of_day', 'agent_experience']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, covariates):
    for treat, grp in df.groupby('ai_assisted'):
        label = 'AI assisted' if treat else 'Human only'
        grp[col].hist(ax=ax, alpha=0.5, bins=20, label=label, density=True)
    ax.set_title(col)
    ax.legend()
plt.tight_layout()
plt.show()

## 5. Pre-matching covariate balance

In [ ]:
bt = balance_table(df, treatment_col='ai_assisted', covariates=covariates)
bt.round(3)

In [ ]:
# Love plot — SMD before matching
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(bt['covariate'], bt['smd'].abs())
ax.axvline(0.1, color='red', linestyle='--', label='|SMD| = 0.1 threshold')
ax.set_xlabel('|Standardised Mean Difference|')
ax.set_title('Covariate balance before matching')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Correlation heatmap

In [ ]:
cols = covariates + outcomes + ['ai_assisted']
corr = df[cols].corr().round(2)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Correlation matrix')
plt.tight_layout()
plt.show()

## 7. Time series — daily conversation volume

In [ ]:
daily = (
    df.set_index('created_at')
    .groupby([pd.Grouper(freq='W'), 'ai_assisted'])
    .size()
    .unstack(fill_value=0)
    .rename(columns={0: 'Human only', 1: 'AI assisted'})
)

daily.plot(figsize=(12, 4), title='Weekly conversation volume by treatment')
plt.ylabel('Conversations')
plt.tight_layout()
plt.show()